In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
# Missing values per column
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [4]:
# Churn class balance
df['Churn'].value_counts()
df['Churn'].value_counts(normalize=True) * 100

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

In [5]:
# Find rows where TotalCharges is blank/whitespace
blank_mask = df['TotalCharges'].str.strip() == ''
print("Blank TotalCharges rows:", blank_mask.sum())
df[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

Blank TotalCharges rows: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [6]:
# Convert TotalCharges to numeric, coercing blanks to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Impute: for tenure=0 rows, TotalCharges = MonthlyCharges
df.loc[df['TotalCharges'].isnull(), 'TotalCharges'] = df.loc[df['TotalCharges'].isnull(), 'MonthlyCharges']

# Confirm no more missing values and correct dtype
print(df['TotalCharges'].isnull().sum())
print(df['TotalCharges'].dtype)

0
float64


In [7]:
df.to_csv('../data/processed/telco_churn_cleaned.csv', index=False)
print("Saved:", df.shape)

Saved: (7043, 21)


In [8]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp (which becomes val+test)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['Churn'],
    random_state=42
)

# Second split: divide the 30% temp into 15% val, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['Churn'],
    random_state=42
)

print("Train:", train_df.shape, "Churn %:", train_df['Churn'].value_counts(normalize=True)['Yes']*100)
print("Val:  ", val_df.shape, "Churn %:", val_df['Churn'].value_counts(normalize=True)['Yes']*100)
print("Test: ", test_df.shape, "Churn %:", test_df['Churn'].value_counts(normalize=True)['Yes']*100)

Train: (4930, 21) Churn %: 26.531440162271807
Val:   (1056, 21) Churn %: 26.515151515151516
Test:  (1057, 21) Churn %: 26.584673604541152


In [9]:
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print("Saved train, val, test splits")

Saved train, val, test splits


In [10]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())

Duplicate rows: 0
Duplicate customerIDs: 0


In [11]:
from sklearn.preprocessing import LabelEncoder

# Drop customerID - it's an identifier, not a predictive feature
train_model_df = train_df.drop(columns=['customerID']).copy()
val_model_df = val_df.drop(columns=['customerID']).copy()

# Encode target
train_model_df['Churn'] = train_model_df['Churn'].map({'Yes': 1, 'No': 0})
val_model_df['Churn'] = val_model_df['Churn'].map({'Yes': 1, 'No': 0})

# Encode all remaining categorical columns
categorical_cols = train_model_df.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", categorical_cols)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_model_df[col] = le.fit_transform(train_model_df[col])
    val_model_df[col] = le.transform(val_model_df[col])
    encoders[col] = le

X_train = train_model_df.drop(columns=['Churn'])
y_train = train_model_df['Churn']
X_val = val_model_df.drop(columns=['Churn'])
y_val = val_model_df['Churn']

print("X_train:", X_train.shape, "X_val:", X_val.shape)

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
X_train: (4930, 19) X_val: (1056, 19)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]

print("F1-score:", f1_score(y_val, y_pred))
print("AUC-ROC:", roc_auc_score(y_val, y_proba))
print()
print(classification_report(y_val, y_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

F1-score: 0.6131386861313868
AUC-ROC: 0.8446681701030927

              precision    recall  f1-score   support

           0       0.86      0.87      0.86       776
           1       0.63      0.60      0.61       280

    accuracy                           0.80      1056
   macro avg       0.74      0.74      0.74      1056
weighted avg       0.80      0.80      0.80      1056


Confusion Matrix:
[[676 100]
 [112 168]]


In [15]:
import mlflow
mlflow.set_tracking_uri("file:../mlruns")

In [16]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("telco-churn-prediction")

with mlflow.start_run(run_name="baseline_logistic_regression"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("features", list(X_train.columns))

    mlflow.log_metric("f1_score", f1_score(y_val, y_pred))
    mlflow.log_metric("auc_roc", roc_auc_score(y_val, y_proba))
    mlflow.log_metric("accuracy", (y_pred == y_val).mean())

    mlflow.sklearn.log_model(model, "model")

    print("Run logged. Run ID:", mlflow.active_run().info.run_id)

2026/09/17 13:20:24 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-prediction' does not exist. Creating a new experiment.
2026/09/17 13:20:28 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run logged. Run ID: ecb46e0502ae4357964b2d62f633cd40


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_val)
rf_proba = rf_model.predict_proba(X_val)[:, 1]

print("F1-score:", f1_score(y_val, rf_pred))
print("AUC-ROC:", roc_auc_score(y_val, rf_proba))
print()
print(classification_report(y_val, rf_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val, rf_pred))

F1-score: 0.5664062500000001
AUC-ROC: 0.8377002025036819

              precision    recall  f1-score   support

           0       0.84      0.89      0.86       776
           1       0.62      0.52      0.57       280

    accuracy                           0.79      1056
   macro avg       0.73      0.70      0.71      1056
weighted avg       0.78      0.79      0.78      1056


Confusion Matrix:
[[689  87]
 [135 145]]


In [18]:
rf_model_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model_balanced.fit(X_train, y_train)

rf_bal_pred = rf_model_balanced.predict(X_val)
rf_bal_proba = rf_model_balanced.predict_proba(X_val)[:, 1]

print("F1-score:", f1_score(y_val, rf_bal_pred))
print("AUC-ROC:", roc_auc_score(y_val, rf_bal_proba))
print()
print(classification_report(y_val, rf_bal_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val, rf_bal_pred))

F1-score: 0.6275115919629057
AUC-ROC: 0.8360985824742269

              precision    recall  f1-score   support

           0       0.89      0.79      0.84       776
           1       0.55      0.72      0.63       280

    accuracy                           0.77      1056
   macro avg       0.72      0.76      0.73      1056
weighted avg       0.80      0.77      0.78      1056


Confusion Matrix:
[[612 164]
 [ 77 203]]


In [19]:
# Log RF (default) run
with mlflow.start_run(run_name="random_forest_default"):
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "none")
    mlflow.log_metric("f1_score", f1_score(y_val, rf_pred))
    mlflow.log_metric("auc_roc", roc_auc_score(y_val, rf_proba))
    mlflow.sklearn.log_model(rf_model, "model")

# Log RF (balanced) run
with mlflow.start_run(run_name="random_forest_balanced"):
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("f1_score", f1_score(y_val, rf_bal_pred))
    mlflow.log_metric("auc_roc", roc_auc_score(y_val, rf_bal_proba))
    mlflow.sklearn.log_model(rf_model_balanced, "model")

print("Both RF runs logged")

2026/09/17 13:50:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/09/17 13:50:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Both RF runs logged
